# MPC có tự đi được G1 ở 1,5 m/s không? (Colab, GPU)

Bước kiểm tra **trước khi** so sánh PPO, MPC-DAgger và MPC-Inject trên G1. Hai phương pháp
MPC cần một "thầy" MPC làm được task; nếu MPC không đi được thì không có gì để học.

Cho MPC (planner lấy mẫu MPPI / MPOPI) tự điều khiển G1 trên `Mjlab-Velocity-Flat-Unitree-G1`
với lệnh đi thẳng 0,5 / 1,0 / 1,5 m/s, so với mốc "đứng yên" (hành động bằng 0), đo vận tốc
thực tế, sai số và số lần ngã, và quay video.

**Tiêu chí đạt (đặt trước, `docs/mpc_g1_velocity.md`):** ít nhất một cấu hình planner đạt vận
tốc trung bình ≥ 1,2 m/s ở lệnh 1,5 m/s và ≤ 0,25 lần ngã mỗi env trong 5 giây.

| Planner | Mẫu × vòng | Tầm nhìn (bước, 50 Hz) | Độ lớn nhiễu | Nhiễu mượt |
|---|---|---|---|---|
| P1 MPPI | 64 × 1 | 20 (0,4 s) | 0,3 | không |
| P2 MPPI | 64 × 1 | 20 (0,4 s) | 0,1 | 4 điểm |
| P3 MPPI | 128 × 1 | 40 (0,8 s) | 0,2 | 4 điểm |
| P4 MPOPI | 32 × 4 | 40 (0,8 s) | 0,2 | 4 điểm |

**Trước khi chạy:** *Runtime → Change runtime type → GPU*. Nên dùng L4 hoặc A100 nếu có;
T4 cũng chạy được nhưng chậm hơn. Mục 5 in thời gian lập kế hoạch mỗi bước sau 25 bước
đầu; nếu quá chậm, giảm `NUM_ENVS` hoặc `STEPS`.

## 0. Cấu hình

In [ ]:
#@title Cấu hình chung
REPO_URL = "https://github.com/TamasTran/mjlab_MPOPI.git"  #@param {type:"string"}
BRANCH = "mpc-stage1"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
NUM_ENVS = 8  #@param {type:"integer"}
STEPS = 250  #@param {type:"integer"}
SPEEDS = "0.5 1.0 1.5"  #@param {type:"string"}
#@markdown Planner cần chạy (cách nhau bằng dấu cách):
PLANNERS = "P1 P2 P3 P4"  #@param {type:"string"}
RECORD_VIDEO = True  #@param {type:"boolean"}

import os

REPO_DIR = "/content/mjlab_MPOPI"
if SAVE_TO_DRIVE:
  from google.colab import drive

  drive.mount("/content/drive")
  OUT_ROOT = "/content/drive/MyDrive/mpc_g1_check"
else:
  OUT_ROOT = "/content/mpc_g1_check"
os.makedirs(OUT_ROOT, exist_ok=True)
print("Kết quả lưu tại:", OUT_ROOT)

In [ ]:
#@title Hàm tiện ích: chạy lệnh và in log trực tiếp
import subprocess


def run(cmd: str, cwd: str | None = None) -> None:
  """Run a shell command, stream its output, and raise if it fails."""
  print(f"$ {cmd}", flush=True)
  proc = subprocess.Popen(
    cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env={**os.environ},
  )
  assert proc.stdout is not None
  for line in proc.stdout:
    if "[INFO]" in line or line.startswith(("|", "+")) or not line.strip():
      continue  # Env construction tables.
    print(line, end="", flush=True)
  if proc.wait() != 0:
    raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Lấy code

Nhánh `mpc-stage1` phải được push bản mới nhất (có `state_sync.py` và `eval_mpc_velocity.py`).

In [ ]:
import shutil

if os.path.exists(REPO_DIR):
  shutil.rmtree(REPO_DIR)
run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
run("git log --oneline -1", cwd=REPO_DIR)
missing = [
  f for f in ("src/mjlab/mpc/state_sync.py", "scripts/mpc/eval_mpc_velocity.py")
  if not os.path.exists(f"{REPO_DIR}/{f}")
]
if missing:
  raise RuntimeError(f"Code trên GitHub đã cũ (thiếu {missing}). Hãy push nhánh mpc-stage1 rồi chạy lại.")
print("OK")

## 3. Cài đặt môi trường

Lần đầu mất khoảng 5–10 phút.

In [ ]:
run("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
run("uv sync", cwd=REPO_DIR)
run(
  "uv run python -c \"import torch; print('torch', torch.__version__, 'cuda', "
  "torch.cuda.is_available(), torch.cuda.get_device_name(0))\"",
  cwd=REPO_DIR,
)

## 4. Test trên GPU

Gồm test so khớp của planner trên G1: sau khi chép trạng thái, planner phải cho reward giống
env thật (sai khác ≤ 1e-3). Nếu ô này lỗi thì **dừng lại**.

In [ ]:
run(
  "uv run pytest tests/test_mpc_state_sync.py tests/test_mpc_sampling.py -q -p no:warnings",
  cwd=REPO_DIR,
)

## 5. Chạy đánh giá

Mỗi planner chạy 3 tốc độ; kết quả lưu vào `OUT_ROOT/<planner>.json`, video vào
`OUT_ROOT/video/<planner>/`. Planner nào đã có file JSON thì bỏ qua (chạy lại được sau khi
Colab ngắt). Mốc "đứng yên" chạy cùng P1.

In [ ]:
PLANNER_ARGS = {
  "P1": "--mpc.num-samples 64 --mpc.iterations 1 --mpc.horizon 20 --mpc.noise-std 0.3",
  "P2": "--mpc.num-samples 64 --mpc.iterations 1 --mpc.horizon 20 --mpc.noise-std 0.1 --mpc.num-knots 4",
  "P3": "--mpc.num-samples 128 --mpc.iterations 1 --mpc.horizon 40 --mpc.noise-std 0.2 --mpc.num-knots 4",
  "P4": "--mpc.num-samples 32 --mpc.iterations 4 --mpc.horizon 40 --mpc.noise-std 0.2 --mpc.num-knots 4",
}
import time

for name in PLANNERS.split():
  out = f"{OUT_ROOT}/{name}.json"
  if os.path.exists(out):
    print(f"{name}: đã có kết quả, bỏ qua")
    continue
  controllers = "zero mpc" if name == "P1" else "mpc"
  video = f"--video-dir {OUT_ROOT}/video/{name}" if RECORD_VIDEO else ""
  t0 = time.time()
  run(
    f"uv run python scripts/mpc/eval_mpc_velocity.py --device cuda:0 --num-envs {NUM_ENVS} "
    f"--steps {STEPS} --speeds {SPEEDS} --controllers {controllers} {PLANNER_ARGS[name]} "
    f"--out {out} {video}",
    cwd=REPO_DIR,
  )
  print(f"===== {name} xong sau {(time.time() - t0) / 60:.1f} phút\n")

## 6. Kết quả và kết luận

In [ ]:
import glob
import json

rows = []
for path in sorted(glob.glob(f"{OUT_ROOT}/P*.json")):
  name = os.path.basename(path).removesuffix(".json")
  for r in json.load(open(path))["results"]:
    rows.append({"planner": name if r["controller"] == "mpc" else "đứng yên", **r})
seen = set()
print(f"{'planner':>9} {'lệnh':>5} {'vận tốc':>8} {'|sai số|':>9} {'ngã/env':>8} {'reward/bước':>12} {'giây/bước':>10}")
for r in sorted(rows, key=lambda r: (r["command"], r["planner"])):
  key = (r["planner"], r["command"])
  if key in seen:
    continue
  seen.add(key)
  plan = f"{r['plan_seconds_per_step']:.2f}" if "plan_seconds_per_step" in r else "-"
  print(f"{r['planner']:>9} {r['command']:>5.1f} {r['speed']:>8.2f} {r['abs_error']:>9.2f} "
        f"{r['falls_per_env']:>8.2f} {r['reward_per_step']:>12.4f} {plan:>10}")

passed = [r["planner"] for r in rows if r["controller"] == "mpc" and r["command"] == 1.5
          and r["speed"] >= 1.2 and r["falls_per_env"] <= 0.25]
print("\nTiêu chí (≥ 1,2 m/s ở lệnh 1,5 m/s, ≤ 0,25 lần ngã/env):",
      f"ĐẠT với {passed}" if passed else "KHÔNG planner nào đạt")

## 7. Xem video (env đầu tiên)

In [ ]:
from IPython.display import HTML, Video, display

for path in sorted(glob.glob(f"{OUT_ROOT}/video/*/*.mp4")):
  display(HTML(f"<b>{os.path.relpath(path, OUT_ROOT)}</b>"))
  display(Video(path, embed=True, width=480))

## 8. Tải kết quả về

In [ ]:
shutil.make_archive("/content/mpc_g1_check_results", "zip", OUT_ROOT)
from google.colab import files

files.download("/content/mpc_g1_check_results.zip")